## WEEK 7 – ASSIGNMENT 

##### Objective:
To demonstrate incremental data processing in Delta Lake by loading customer data, cleaning the dataset, performing MERGE operations to update and insert records, and validating the final Delta table.

#### Step 1: Load the dataset into a Delta table

Reading the CSV file

In [0]:
master_df = spark.read.option("header","true").option("inferSchema","true").csv("/Volumes/workspace/default/delta_assignment/customer_master.csv")

In [0]:
display(master_df)

customer_id,name,email,city,signup_date,status
1148,Qabil Sura,qabil.sura26@example.com,Chennai,2024-02-23,active
1418,Samar Oak,samar.oak236@example.com,Vadodara,2025-04-18,inactive
1472,Bimala Barad,bimala.barad135@example.com,Hyderabad,2023-07-21,active
2064,Diya Anand,diya.anand891@example.com,Ranchi,2023-07-24,active
1128,Ladli Narang,ladli.narang279@example.com,Lucknow,2024-02-22,active
1084,Nisha Kothari,nisha.kothari479@example.com,Raipur,2024-04-21,active
1791,Siddharth Hans,siddharth.hans18@example.com,Chandigarh,null,active
2271,Udyati Dalal,udyati.dalal996@example.com,Pune,2025-09-05,active
1723,Sneha Krish,sneha.krish76@example.com,Pune,2023-10-20,inactive
2091,Mekhala Viswanathan,mekhala.viswanathan437@example.com,Visakhapatnam,2025-10-04,active


In [0]:
master_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)



Saving it as a Delta table

In [0]:
master_df.write.format("delta").mode("overwrite").saveAsTable("customer_master_delta")

In [0]:
display(master_df.limit(10))

customer_id,name,email,city,signup_date,status
1148,Qabil Sura,qabil.sura26@example.com,Chennai,2024-02-23,active
1418,Samar Oak,samar.oak236@example.com,Vadodara,2025-04-18,inactive
1472,Bimala Barad,bimala.barad135@example.com,Hyderabad,2023-07-21,active
2064,Diya Anand,diya.anand891@example.com,Ranchi,2023-07-24,active
1128,Ladli Narang,ladli.narang279@example.com,Lucknow,2024-02-22,active
1084,Nisha Kothari,nisha.kothari479@example.com,Raipur,2024-04-21,active
1791,Siddharth Hans,siddharth.hans18@example.com,Chandigarh,null,active
2271,Udyati Dalal,udyati.dalal996@example.com,Pune,2025-09-05,active
1723,Sneha Krish,sneha.krish76@example.com,Pune,2023-10-20,inactive
2091,Mekhala Viswanathan,mekhala.viswanathan437@example.com,Visakhapatnam,2025-10-04,active


#### Step 2: Data Cleaning

In [0]:
customer_df=spark.table("customer_master_delta")

In [0]:
print("Rows before cleaning the dataset:", customer_df.count())

Rows before cleaning the dataset: 1339


Checking for null values

In [0]:
from pyspark.sql.functions import col,count,when
customer_df.select([count(when(col(c).isNull(), c)).alias(c)for c in customer_df.columns]).show()

+-----------+----+-----+----+-----------+------+
|customer_id|name|email|city|signup_date|status|
+-----------+----+-----+----+-----------+------+
|         20|  27|   65|  54|         43|     0|
+-----------+----+-----+----+-----------+------+



Handling null values

In [0]:
customer_df=customer_df.fillna({ "city": "Unknown","status": "Active"})

Checking for duplicate records

In [0]:
customer_df.count()


1339

In [0]:
customer_df.dropDuplicates().count()

1300

Removing duplicates Rows

In [0]:
customer_df=customer_df.dropDuplicates()

In [0]:
customer_df=customer_df.dropDuplicates(["customer_id"])

In [0]:
customer_df.count()

1379

Saving the cleaned Delta table

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("customer_master_cleaned")

#### Step 3: Loading the Incremental Dataset

In [0]:
incremental_df=spark.read.option("header","true").option("inferSchema","true").csv("/Volumes/workspace/default/delta_assignment/customer_incremental.csv")

In [0]:
display(incremental_df)

customer_id,name,email,city,signup_date,status
2366,Kabir Peri,kabir.peri974@example.com,Surat,2024-07-17,active
1845,Thomas Borah,thomas.borah45@newmail.com,Pune,2024-12-08,active
2178,Lohit Suresh,lohit.suresh809@newmail.com,Jaipur,2025-06-26,active
2057,Zayan Hans,zayan.hans187@newmail.com,Pune,2023-08-18,active
1513,Bhanumati Saha,bhanumati.saha573@newmail.com,Lucknow,2025-08-09,active
2146,Gautam Kalla,gautam.kalla391@newmail.com,Vadodara,2025-12-22,active
1332,Bhavya Dass,bhavya.dass189@newmail.com,Hyderabad,2023-12-06,active
2254,Yashoda Chauhan,yashoda.chauhan645@newmail.com,Bengaluru,2024-09-28,active
2077,Samesh Banerjee,samesh.banerjee591@newmail.com,Nagpur,2025-07-09,active
1496,Sarthak Buch,sarthak.buch185@newmail.com,Patna,2025-05-21,active


In [0]:
incremental_df.count()

310

In [0]:
incremental_df.write.format("delta").mode("overwrite").saveAsTable("customer_incremental_delta")

In [0]:
display(spark.table("customer_incremental_delta"))

customer_id,name,email,city,signup_date,status
2366,Kabir Peri,kabir.peri974@example.com,Surat,2024-07-17,active
1845,Thomas Borah,thomas.borah45@newmail.com,Pune,2024-12-08,active
2178,Lohit Suresh,lohit.suresh809@newmail.com,Jaipur,2025-06-26,active
2057,Zayan Hans,zayan.hans187@newmail.com,Pune,2023-08-18,active
1513,Bhanumati Saha,bhanumati.saha573@newmail.com,Lucknow,2025-08-09,active
2146,Gautam Kalla,gautam.kalla391@newmail.com,Vadodara,2025-12-22,active
1332,Bhavya Dass,bhavya.dass189@newmail.com,Hyderabad,2023-12-06,active
2254,Yashoda Chauhan,yashoda.chauhan645@newmail.com,Bengaluru,2024-09-28,active
2077,Samesh Banerjee,samesh.banerjee591@newmail.com,Nagpur,2025-07-09,active
1496,Sarthak Buch,sarthak.buch185@newmail.com,Patna,2025-05-21,active


#### Step 4: Apply the MERGE Operation

In [0]:
%sql
merge into customer_master_cleaned as target
using customer_incremental_delta as source
on target.customer_id=source.customer_id
when matched then
update set target.name=source.name,target.city=source.city,target.email=source.email,
    target.signup_date=source.signup_date,target.status=source.status
when not matched then
insert (customer_id,name,city,email,signup_date,status)
values (source.customer_id,source.name,source.city,source.email,
    source.signup_date,source.status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
310,310,0,0


#### Step 5: Validate the MERGE Results

In [0]:
final_df = spark.table("customer_master_cleaned")

Checking the total number of records

In [0]:
display(final_df.count())

1379

Checking for duplicate customer_ids

In [0]:
from pyspark.sql.functions import col, count
duplicates=(final_df.groupBy("customer_id").agg(count("*").alias("count")).filter(col("count")>1)
)
display(duplicates)

customer_id,count


Verifing newly inserted records

In [0]:
display(final_df.filter(col("customer_id")=="2366"))

customer_id,name,email,city,signup_date,status
2366,Kabir Peri,kabir.peri974@example.com,Surat,2024-07-17,active


#### Step 6: Displaying the Final Dataset and Summary

In [0]:
display(final_df.limit(10))

customer_id,name,email,city,signup_date,status
1918,Hemani Deo,hemani.deo243@example.com,Lucknow,2026-05-23,inactive
1160,Utkarsh Patel,utkarsh.patel550@example.com,Kolkata,null,inactive
2185,Vinaya Rana,vinaya.rana73@example.com,Jaipur,2025-06-06,active
1256,Jason Sha,jason.sha870@example.com,Surat,2023-10-17,inactive
2246,Yasti Dutta,yasti.dutta13@example.com,Jaipur,2024-08-18,active
1736,Leela Devan,null,Thiruvananthapuram,2024-09-30,active
2191,Chaitanya Kaur,chaitanya.kaur169@example.com,Visakhapatnam,2024-02-09,active
1580,Daksha Ahuja,daksha.ahuja831@example.com,Kolkata,2026-04-18,inactive
1926,Jagrati Sethi,jagrati.sethi327@example.com,Ranchi,2023-10-29,active
1342,Vivaan Walla,vivaan.walla323@example.com,Ahmedabad,2024-03-22,active


In [0]:
final_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)



Summary

In [0]:
print("Assignment Summary")
print("Loaded customer master dataset into Delta table.")
print("Performed data cleaning (handled nulls and removed duplicates).")
print("Loaded incremental customer dataset.")
print("Applied Delta Lake MERGE operation.")
print("Updated existing customer records.")
print("Inserted new customer records.")
print("Validated the final Delta table.")


Assignment Summary
Loaded customer master dataset into Delta table.
Performed data cleaning (handled nulls and removed duplicates).
Loaded incremental customer dataset.
Applied Delta Lake MERGE operation.
Updated existing customer records.
Inserted new customer records.
Validated the final Delta table.
